
### Purpose
This notebook trains an **Isolation Forest** model to detect anomalous sales behavior and creates anomaly-related features for downstream use.
<br>
📝 Language: Technical documentation is maintained in English to ensure consistency and ease of maintenance. Bilingual support (Portuguese/English) is available exclusively within the notebooks.
<br> --- --- --- <br>
📝 Idioma: A documentação é mantida em inglês para garantir consistência técnica e evitar retrabalho na documentação. Isto aplica-se somente aos notebooks.

##### ⚙️The project follows this logic:
**raw data -> data quality -> feature engineering -> anomaly detection -> baseline model -> hybrid model -> performance benchmarking**

##### ⚙️Execution order
📝01_data_loading.ipynb -> 📝02_data_quality.ipynb -> 📝03_feature_engineering.ipynb -> 📝04_anomaly_detection.ipynb -> 
📝05_BaseLine.ipynb -> 📝06_HybridModel.ipynb -> 📝07_Interpretability_SHAP_Analysis.ipynb -> 📝08_performance_benchmarking.ipynb  
 

### Main Components
- `ParquetRepository`
- `IsolationForestAnalyzer`
- `DATA_FEATURES`
- `MODELS_ANOMALYD`

### Input
- 📦`beverage_sales_feature.parquet`

### What the Notebook Does
- Loads the engineered feature dataset
- Converts `Order_Date` to datetime
- Splits data by year:
  - Train: 2021 and 2022
  - Test / future period: 2023
- Creates an `IsolationForestAnalyzer` with:
  - `random_state=42`
  - `cv=3`
  - `n_jobs=-1`
  - `verbose=1`
- Fits the model on the training period only
- Prints the best parameters
- Saves the trained model as a `.joblib` file
- Saves the best parameters as a JSON file
- Runs prediction on the feature dataset
- Selects anomaly-related columns, including:
  - `anomaly_flag`
  - `anomaly_label`
  - `anomaly_score`
  - `if_qty_signal`
  - `if_sales_signal`
  - `if_discount_signal`
  - `if_ticket_signal`
- Prints an anomaly summary
<br> --- --- --- <br>

- Carrega o dataset com features
- Converte `Order_Date` para datetime
- Divide os dados por ano:
  - Treino: 2021 e 2022
  - Teste / período futuro: 2023
- Cria um `IsolationForestAnalyzer` com:
  - `random_state=42`
  - `cv=3`
  - `n_jobs=-1`
  - `verbose=1`
- Treina o modelo apenas no período de treino
- Exibe os melhores parâmetros
- Salva o modelo treinado em arquivo `.joblib`
- Salva os melhores parâmetros em JSON
- Executa predição no dataset de features
- Seleciona colunas relacionadas a anomalias, incluindo:
  - `anomaly_flag`
  - `anomaly_label`
  - `anomaly_score`
  - `if_qty_signal`
  - `if_sales_signal`
  - `if_discount_signal`
  - `if_ticket_signal`
- Exibe um resumo das anomalias

### Output
- Serialized model: `isolation_forest_analyzer.joblib`
- Parameters file: `isolation_forest_best_params.json`
- Anomaly predictions in notebook memory/output
<br> --- --- --- <br>

- Modelo serializado: `isolation_forest_analyzer.joblib`
- Arquivo de parâmetros: `isolation_forest_best_params.json`
- Predições de anomalia disponíveis na memória/saída do notebook

### Why This Step Matters
- Detects unusual patterns without requiring labels
- Adds useful anomaly signals that can later be consumed by supervised models
- Supports hybrid modeling strategies
<br> --- --- --- <br>

- Detecta padrões incomuns sem precisar de rótulos
- Cria sinais úteis de anomalia que podem ser usados por modelos supervisionados depois
- Dá suporte a uma estratégia híbrida de modelagem

### 🔮 Leakage Prevention
🚨Important rule:
> To avoid data leakage, the Isolation Forest model is trained only on 2021–2022 data, leaving 2023 for testing.
<br> --- --- --- <br>

🚨Regra importante:
> Para evitar data leakage, o modelo Isolation Forest é treinado apenas com dados de 2021 a 2022, deixando 2023 para testes.
<br>
### 🚨Memory Error in GridSearchCV:
  To avoid memory overflow, the cross-validation parameter (cv) was set to 3 (default).  
  The notebook was executed on a Ryzen 7 processor with 32GB of RAM. If your 
  machine has less memory, consider lowering this value to 2 or 1 in the 
  initialization parameters of the "IsolationForestAnalyzer" class (located 
  in the 'src/models' folder).

  Error description: 
  MemoryError: Unable to allocate 1.40 MiB for an array with shape (2, 182987) and data type int32
<br> --- --- --- <br>
- Erro de Memória no GridSearchCV:
  Para evitar o estouro de memória (memory overflow), o parâmetro de validação cruzada (cv) foi definido como 3 (default). 
  O notebook foi executado em um processador Ryzen 7 com 32GB de RAM. Se a sua máquina tiver menos memória, 
  considere reduzir esse valor para 2 ou 1 nos parâmetros de inicialização da classe "IsolationForestAnalyzer" (localizada na pasta 'src/models').

  Descrição do erro: 
  MemoryError: Unable to allocate 1.40 MiB for an array with shape (2, 182987) and data type int32


In [8]:
%load_ext autoreload
%autoreload 2

import os
import glob
from pathlib import Path
import sys
import pyarrow
import pandas as pd


PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import DATA_PROCESSED, DATA_FEATURES, MODELS_ANOMALYD,DATA_EXPORTS
from src.loaders.csv_loader import CSVLoader
from src.repository.parquet_repository import ParquetRepository

from src.models.IsolationForest import IsolationForestAnalyzer


csvLoader = CSVLoader()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
repo = ParquetRepository(DATA_FEATURES)

In [10]:
df_load = repo.load("beverage_sales_feature.parquet")

[OK] Arquivo carregado: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\features\beverage_sales_feature.parquet


In [11]:
print (df_load.dtypes)

Order_Date                  datetime64[ns]
Category                            object
Product                             object
Region                              object
quantity_sum                       float64
total_price_sum                    float64
unit_price_mean                    float64
discount_mean                      float64
order_count                        float64
customer_count                     float64
avg_ticket                         float64
day_of_week                          int32
month                                int32
year                                 int32
is_weekend                           int64
quantity_sum_mean_7d               float64
quantity_sum_std_7d                float64
quantity_sum_sum_7d                float64
total_price_sum_mean_7d            float64
total_price_sum_std_7d             float64
total_price_sum_mean_30d           float64
unit_price_mean_mean_7d            float64
discount_mean_mean_14d             float64
quantity_vs

# Atencao
Para evitar Data Leakage, o modelo Isolation Forest é treinado apenas para o periodo de 2021 a 2022 deixando o ano 2023 para testes

In [12]:
df_load["Order_Date"] = pd.to_datetime(df_load["Order_Date"])

df_train_if = df_load[df_load["Order_Date"].dt.year.isin([2021, 2022])].copy()
df_test_if = df_load[df_load["Order_Date"].dt.year == 2023].copy()

In [13]:
if_analyzer = IsolationForestAnalyzer(
    random_state=42,
    cv=3,
    n_jobs=-1,
    verbose=1
)

if_analyzer.fit(df_train_if)



Fitting 3 folds for each of 72 candidates, totalling 216 fits


In [14]:
print(if_analyzer.get_best_params())


{'model__contamination': 0.01, 'model__max_features': 1.0, 'model__max_samples': 512, 'model__n_estimators': 100}


In [15]:
model_path = if_analyzer.save_model(
    folder_path=MODELS_ANOMALYD,
    file_name="isolation_forest_analyzer.joblib"
)

In [16]:
model_path = if_analyzer.save_best_params_json(
    folder_path=MODELS_ANOMALYD,
    file_name="isolation_forest_best_params.json"
)

In [17]:
df_if = if_analyzer.predict(df_load)

cols_anomaly_output = [
    "Order_Date",
    "Product",
    "Region",
    "anomaly_flag",
    "anomaly_label",
    "anomaly_score",
    "if_qty_signal",
    "if_sales_signal",
    "if_discount_signal",
    "if_ticket_signal"
]

df_if_output = df_if[cols_anomaly_output].copy()



In [18]:
summary_if = if_analyzer.anomaly_summary(df_if)
print(summary_if.head(20))

            Product                  Region  total_rows  anomaly_count  \
662  Veuve Clicquot                  Hessen        1094            237   
666  Veuve Clicquot         Rheinland-Pfalz        1094            227   
663  Veuve Clicquot  Mecklenburg-Vorpommern        1094            217   
660  Veuve Clicquot                  Bremen        1094            214   
656  Veuve Clicquot       Baden-Württemberg        1094            199   
661  Veuve Clicquot                 Hamburg        1094            196   
420  Moët & Chandon                  Bremen        1094            190   
423  Moët & Chandon  Mecklenburg-Vorpommern        1094            190   
667  Veuve Clicquot                Saarland        1094            189   
422  Moët & Chandon                  Hessen        1094            188   
669  Veuve Clicquot          Sachsen-Anhalt        1094            181   
659  Veuve Clicquot             Brandenburg        1094            177   
665  Veuve Clicquot     Nordrhein-West

### Export parquet to API project

- To finalize the model, a compressed Parquet file is generated to be used in the API project. It is only required in the Beverage-Sales API project because the API relies on historical data to return responses, due to the use of a Sliding Window.
Location (data/exports)
<br> --- --- --- <br>
Para finalizar o pipeline, um arquivo Parquet compactado é gerado para abastecer a API. Ele é necessário exclusivamente no projeto Beverage-Sales, já que a API utiliza dados históricos para responder às requisições devido à implementação da Janela Deslizante (Sliding Window)
Localizacao (data/exports)
- 📦`anomaly_predictions_api.parquet`

In [21]:
repo.set_base_path(DATA_EXPORTS)
repo.save(df_if, "anomaly_predictions_api.parquet",compression="zstd")

[OK] Arquivo salvo em: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\exports\anomaly_predictions_api.parquet
